# 01 — Détecter les connexions SSH suspectes

**IA-01 · Jour 3 · Hexalp**

L'équipe supervision a extrait deux semaines de journaux SSH du bastion, du 14 au 27 septembre 2026. Chaque ligne résume l'activité d'une adresse IP pendant une heure. Un analyste a étiqueté chaque ligne `normal` ou `attaque`.

Vous allez suivre la chaîne complète vue en 3.2 :

| Partie | Étape | Séquence du kit |
|---|---|---|
| A | Explorer les observations | 3.5 |
| B | Nettoyer et préparer les caractéristiques | 3.5 |
| C | Entraîner et évaluer un classifieur supervisé | 3.6 |
| D | Détecter des anomalies sans étiquettes | 3.7 |

Remplacez chaque `___`. Chaque ✍️ appelle une réponse dans votre carnet de bord.

⚠️ Le modèle ne bloque rien. Il produit des alertes qu'un analyste doit qualifier.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score

GRAINE = 42      # graine du hasard : avec la même graine, on obtient les mêmes résultats à chaque exécution

---
## Partie A — Explorer les observations

In [ ]:
df = pd.read_csv(___)
print(df.shape)
df.head()

| Colonne | Signification |
|---|---|
| `date`, `heure` | Jour et heure (0 à 23) de l'observation |
| `ip_source` | Adresse IP à l'origine des connexions |
| `origine` | `interne` (réseau Hexalp) ou `externe` (Internet) |
| `echecs_auth` | Nombre d'authentifications échouées dans l'heure |
| `succes_auth` | Nombre d'authentifications réussies dans l'heure |
| `comptes_distincts` | Nombre de comptes différents utilisés ou testés |
| `duree_session_min` | Durée cumulée des sessions ouvertes, en minutes |
| `octets_sortants_mo` | Volume sorti par les sessions SSH, en Mo (sonde NetFlow) |
| `fail2ban_ban` | 1 si fail2ban a banni l'IP pendant cette heure |
| `etiquette` | La réponse de l'analyste : `normal` ou `attaque` |

In [ ]:
df.info()          # types des colonnes et nombre de valeurs non vides

In [ ]:
df.describe().round(1)     # statistiques de chaque colonne numérique

In [ ]:
print(df["etiquette"].value_counts())
print(df["etiquette"].value_counts(normalize=True).round(3))    # en proportion

✍️ **A1.** Quelle proportion des observations est étiquetée `attaque` ?

✍️ **A2.** Un « modèle » qui répondrait toujours `normal` aurait quel taux de bonnes réponses (*accuracy*) ? Serait-il utile à Nadia ?

In [ ]:
# Le graphique de la mission 3 du J1, sur 1 700 observations
couleurs = df["etiquette"].map({"normal": "tab:blue", "attaque": "tab:red"})
plt.figure(figsize=(8, 5))
plt.scatter(df[___], df[___], c=couleurs, alpha=0.6, s=15)
plt.xlabel("Échecs d'authentification dans l'heure")
plt.ylabel("Comptes distincts")
plt.title("Bleu : normal, rouge : attaque")
plt.show()

✍️ **A3.** Repérez au moins deux zones où des points bleus et rouges se mélangent. Quelles attaques risquent d'être difficiles à repérer avec ces deux seules caractéristiques ?

In [ ]:
# Les observations "normales" les plus spectaculaires : à qui appartiennent-elles ?
df[df["etiquette"] == "normal"].sort_values("echecs_auth", ascending=False).head(5)

✍️ **A4.** Qui est `10.20.1.250` ? Pourquoi l'analyste a-t-il étiqueté ces lignes `normal` ? *(Indice : regardez la date et l'heure.)*

---
## Partie B — Nettoyer et préparer les caractéristiques

Un modèle Scikit-learn n'accepte que des **nombres**, et **aucune case vide**.

In [ ]:
df.isna().sum()       # nombre de cases vides par colonne

In [ ]:
# Où sont les cases vides ?
df[df["octets_sortants_mo"].isna()][["date", "heure"]].value_counts().sort_index().head(20)

✍️ **B1.** La sonde NetFlow est tombée en panne deux fois. Trois options pour les cases vides :

- supprimer les lignes : `df = df.dropna()` ;
- remplacer par 0 : `df["octets_sortants_mo"] = df["octets_sortants_mo"].fillna(0)` ;
- remplacer par la médiane : `df["octets_sortants_mo"] = df["octets_sortants_mo"].fillna(df["octets_sortants_mo"].median())`.

Que perd-on avec chaque option ? Pourquoi « 0 » est-il un mensonge dangereux pour un modèle de détection d'exfiltration ? Choisissez une option et justifiez-la.

In [ ]:
# Votre choix (une seule ligne) :
___

print("Cases vides restantes :", df["octets_sortants_mo"].isna().sum())

In [ ]:
# Transformer le texte en nombre : externe -> 1, interne -> 0
df["externe"] = (df["origine"] == ___).astype(int)
df[["origine", "externe"]].drop_duplicates()

Il faut maintenant choisir les **caractéristiques** (features) données au modèle, et séparer la **réponse** attendue (l'étiquette).

✍️ **B2.** Pour chaque colonne, décidez si elle doit être donnée au modèle. Justifiez en particulier vos choix pour `ip_source`, `date` et `fail2ban_ban`.

In [ ]:
CARACTERISTIQUES = [
    ___
]

X = df[CARACTERISTIQUES]                       # les questions
y = (df["etiquette"] == "attaque").astype(int) # les réponses : 1 = attaque, 0 = normal

print(X.shape, y.sum(), "attaques")

---
## Partie C — Le classifieur supervisé

On met de côté 30 % des observations pour l'évaluation. Le modèle ne les verra **jamais** pendant l'entraînement : c'est votre jeu de test du J1.

`stratify=y` garantit la même proportion d'attaques dans les deux jeux.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=___, stratify=y, random_state=GRAINE
)
print("Entraînement :", X_train.shape, "Test :", X_test.shape)

Un **arbre de décision** fait ce que vous avez fait au J1 : il cherche la caractéristique et le seuil qui séparent le mieux les exemples, puis recommence dans chaque branche. `max_depth` limite le nombre de questions successives.

In [ ]:
arbre = DecisionTreeClassifier(max_depth=___, random_state=GRAINE)
arbre.___(X_train, y_train)          # l'entraînement

predictions = arbre.predict(X_test)        # le modèle répond sur des cas jamais vus
print("Accuracy :", round(accuracy_score(y_test, predictions), 3))

In [ ]:
# Les règles trouvées par la machine
print(export_text(arbre, feature_names=CARACTERISTIQUES))

In [ ]:
plt.figure(figsize=(16, 7))
plot_tree(arbre, feature_names=CARACTERISTIQUES, class_names=["normal", "attaque"], filled=True, fontsize=9)
plt.show()

✍️ **C1.** Traduisez les règles de l'arbre en français, comme vous l'aviez fait au J1. Retrouvez-vous votre règle ? Une branche vous paraît-elle dangereuse ? *(Regardez la branche des connexions internes.)*

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, predictions, display_labels=["normal", "attaque"])
plt.show()
print(classification_report(y_test, predictions, target_names=["normal", "attaque"]))

| Indicateur | La question à laquelle il répond |
|---|---|
| **Précision** (attaque) | Quand le modèle lève une alerte, a-t-il raison ? Faible précision = beaucoup de faux positifs |
| **Rappel** (attaque) | Parmi les vraies attaques, combien en voit-il ? Faible rappel = beaucoup de faux négatifs |

✍️ **C2.** Relevez le nombre de faux positifs et de faux négatifs. Comparez l'accuracy du modèle à celle du « modèle » qui répond toujours `normal` (A2). Quel indicateur Nadia doit-elle regarder ?

In [ ]:
# Examiner les erreurs une par une
test = df.loc[X_test.index].copy()
test["prediction"] = predictions

faux_negatifs = test[(test["prediction"] == 0) & (y_test == 1)]
faux_positifs = test[(test["prediction"] == 1) & (y_test == 0)]
print("Faux négatifs : attaques non détectées")
display(faux_negatifs)
print("Faux positifs : alertes à tort")
display(faux_positifs)

✍️ **C3.** Pour chaque erreur, proposez une explication : quel type d'attaque, ou quel comportement normal, a trompé le modèle ?

In [ ]:
# Expérience 1 : influence de la profondeur
for profondeur in [1, 3, 10, None]:          # None = aucune limite
    modele = DecisionTreeClassifier(max_depth=profondeur, random_state=GRAINE).fit(X_train, y_train)
    p = modele.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    print(f"profondeur={profondeur!s:>4} | score entraînement={modele.score(X_train, y_train):.3f}"
          f" | score test={modele.score(X_test, y_test):.3f} | FP={fp} FN={fn} | nb de feuilles={modele.get_n_leaves()}")

✍️ **C4.** Quelle profondeur retiendriez-vous ? Pensez aux erreurs, mais aussi à votre capacité à expliquer la décision à Nadia (J1, TensorFlow Playground).

In [ ]:
# Expérience 2 : le modèle fait-il mieux que fail2ban ?
# La décision de fail2ban est déjà dans les données : on la compare aux étiquettes, sur le même jeu de test.
decision_f2b = df.loc[X_test.index, ___]
print("fail2ban seul")
print(classification_report(y_test, decision_f2b, target_names=["normal", "attaque"]))

In [ ]:
# Les attaques que l'arbre détecte et que fail2ban laisse passer
test["fail2ban_ban"] = decision_f2b
test[(test["prediction"] == 1) & (test["fail2ban_ban"] == 0) & (y_test == 1)]

✍️ **C5.** Comparez le rappel de fail2ban et celui de votre arbre. Quelles attaques l'arbre détecte-t-il en plus ? À quoi ressemblent-elles ?

✍️ **C6.** Vous avez exclu `fail2ban_ban` des caractéristiques en B2. Si vous l'aviez gardée, qu'aurait risqué d'apprendre le modèle ? Que se passerait-il le jour où la règle fail2ban change ?

---
## Partie D — Détecter des anomalies, sans étiquettes

Dans la vraie vie, personne n'étiquette les journaux. Un **Isolation Forest** ne reçoit que `X`, jamais `y`. Il construit des centaines d'arbres au hasard et mesure combien de coupures il faut pour **isoler** chaque observation : ce qui s'isole vite est inhabituel.

Le paramètre `contamination` indique la part d'observations que le modèle doit signaler.

In [ ]:
iforest = IsolationForest(contamination=___, random_state=GRAINE)
iforest.fit(___)                   # aucune étiquette !

df["score_anomalie"] = iforest.decision_function(X)    # plus le score est bas, plus c'est inhabituel
df["anomalie"] = iforest.predict(X) == -1               # -1 = anomalie, 1 = normal
print(df["anomalie"].sum(), "observations signalées sur", len(df))

In [ ]:
# Les 15 observations les plus inhabituelles
colonnes = ["date", "heure", "ip_source", "origine", "echecs_auth", "succes_auth",
            "comptes_distincts", "octets_sortants_mo", "score_anomalie", "etiquette"]
df.sort_values(___).head(15)[colonnes]

✍️ **D1.** Faites le travail de l'analyste : sans regarder la colonne `etiquette`, qualifiez les 15 lignes (attaque probable, comportement légitime, à investiguer). Puis comparez avec l'étiquette.

In [ ]:
# On ressort les étiquettes uniquement pour évaluer
pd.crosstab(df["anomalie"], df["etiquette"], margins=True)

In [ ]:
# Qui sont les faux positifs ?
df[df["anomalie"] & (df["etiquette"] == "normal")]["ip_source"].value_counts()

✍️ **D2.** Qui sont les faux positifs ? Sont-ils vraiment des anomalies ? Que manquerait-il au modèle pour les reconnaître ?

✍️ **D3.** Quelles attaques passent sous le radar ? Comparez avec le schéma de la partie A.

In [ ]:
# Expérience 3 : influence de la contamination
for c in [0.03, 0.05, 0.10]:
    m = IsolationForest(contamination=c, random_state=GRAINE).fit(X)
    signale = m.predict(X) == -1
    vrais = (signale & (y == 1)).sum()
    faux = (signale & (y == 0)).sum()
    rates = (~signale & (y == 1)).sum()
    print(f"contamination={c:.2f} | alertes={signale.sum():4d} | vraies attaques={vrais:3d}"
          f" | faux positifs={faux:3d} | attaques ratées={rates:3d}")

✍️ **D4.** L'équipe d'astreinte peut traiter 30 alertes par semaine. Quelle contamination choisiriez-vous ? Qu'acceptez-vous de rater ?

✍️ **D5.** Retrouvez les lignes de `10.20.2.14` la nuit du 24 septembre. *(C'est Julien, pendant l'incident WMS du J2.)* Sont-elles signalées ? Que vous apprend cet exemple sur ce qu'un modèle d'anomalies peut voir et ne pas voir ?

In [ ]:
df[df["ip_source"] == ___][colonnes]

---
✅ **Fin du notebook.** Vous avez toutes les briques du script `detection_ssh.py` (séquence 3.8).